# 第 6 章互动实验：已知平移的稠密光流

本实验生成两帧固定随机纹理，真实位移已知，并调用 Farneback 作为观察基线。它不提供 `6-1_前向反向筛选.py` 的实现。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

rng = np.random.default_rng(2026)
texture = rng.normal(110, 32, (120, 160)).astype(np.float32)
texture = cv2.GaussianBlur(texture, (5, 5), 0.8)
cv2.circle(texture, (55, 58), 20, 220, -1)
cv2.rectangle(texture, (95, 28), (135, 88), 40, -1)
frame0 = np.clip(texture, 0, 255).astype(np.uint8)

In [ ]:
def show_flow(dx=5, dy=3, winsize=15):
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    frame1 = cv2.warpAffine(frame0, M, (frame0.shape[1], frame0.shape[0]), borderMode=cv2.BORDER_REFLECT)
    flow = cv2.calcOpticalFlowFarneback(frame0, frame1, None, 0.5, 4, winsize, 3, 5, 1.2, 0)
    margin = max(abs(dx), abs(dy), 10) + 4
    core = flow[margin:-margin, margin:-margin]
    median = np.median(core.reshape(-1, 2), axis=0)
    endpoint_error = np.linalg.norm(core - np.array([dx, dy], np.float32), axis=2).mean()
    step = 12
    yy, xx = np.mgrid[0:frame0.shape[0]:step, 0:frame0.shape[1]:step]
    sampled = flow[::step, ::step]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(frame0, cmap='gray'); axes[0].set_title('前一帧')
    axes[1].imshow(frame1, cmap='gray'); axes[1].set_title(f'后一帧：真实 ({dx}, {dy})')
    axes[2].imshow(frame1, cmap='gray')
    axes[2].quiver(xx, yy, sampled[...,0], sampled[...,1], color='lime', angles='xy', scale_units='xy', scale=1)
    axes[2].set_title(f'中位数 ({median[0]:.2f}, {median[1]:.2f})，EPE={endpoint_error:.2f}')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_flow,
    dx=IntSlider(min=-18, max=18, step=1, value=5),
    dy=IntSlider(min=-12, max=12, step=1, value=3),
    winsize=IntSlider(min=5, max=31, step=2, value=15),
);

## 破坏实验与记录

1. 把位移调到最大，比较小窗口和大窗口。
2. 将后一帧整体变亮后重新计算，观察亮度恒定假设失效。
3. 比较边界与内部区域误差，解释为什么评估时排除了边缘。

把真实位移、估计中位数、EPE 和失败原因写入 `实验记录.md`。